In [1]:
import numpy as np
import sys
if "../" not in sys.path:
  sys.path.append("../") 
from lib.envs.gridworld import GridworldEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [7]:
env = GridworldEnv()

In [26]:
def policy_eval(policy, env, discount_factor=1.0, theta=0.00001):
    """
    Evaluate a policy given an environment and a full description of the environment's dynamics.
    
    Args:
        policy: [S, A] shaped matrix representing the policy.
        env: OpenAI env. env.P represents the transition probabilities of the environment.
            env.P[s][a] is a list of transition tuples (prob, next_state, reward, done).
            env.nS is a number of states in the environment. 
            env.nA is a number of actions in the environment.
        theta: We stop evaluation once our value function change is less than theta for all states.
        discount_factor: Gamma discount factor.
    
    Returns:
        Vector of length env.nS representing the value function.
    """
    # Start with a random (all 0) value function
    V = np.zeros(env.nS)
    while True:
        # # 方法1：同步更新  (需要备份 V_old)
        V_old = np.copy(V)
        V = np.zeros(env.nS)
        for s in range(env.nS):
            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, done in env.P[s][a]:
                    # if done:
                    #     V[s] += action_prob * prob * reward
                    # else:
                    V[s] += action_prob * prob * (reward + discount_factor * V_old[next_state])
        # 计算 abs(V - V_old) 如果都小于 theta 就停止迭代
        if np.allclose(V, V_old, rtol=1e-10, atol=theta):
            break
        
        # # 方法2：异步更新(作者答案)  (不需要备份 V_old)
        # delta = 0
        # # For each state, perform a "full backup"
        # for s in range(env.nS):
        #     v = 0
        #     # Look at the possible next actions
        #     for a, action_prob in enumerate(policy[s]):
        #         # For each action, look at the possible next states...
        #         for  prob, next_state, reward, done in env.P[s][a]:
        #             # Calculate the expected value. Ref: Sutton book eq. 4.6.
        #             v += action_prob * prob * (reward + discount_factor * V[next_state])
        #     # How much our value function changed (across any states) 这里统计了每个状态的变化量，取最大值，如果最大值小于 theta 就停止迭代
        #     delta = max(delta, np.abs(v - V[s]))
        #     V[s] = v
        # # Stop evaluating once our value function change is below a threshold
        # if delta < theta:
        #     break
    return np.array(V)

In [27]:
random_policy = np.ones([env.nS, env.nA]) / env.nA
v = policy_eval(random_policy, env)
v

array([  0.        , -13.99989315, -19.99984167, -21.99982282,
       -13.99989315, -17.99986052, -19.99984273, -19.99984167,
       -19.99984167, -19.99984273, -17.99986052, -13.99989315,
       -21.99982282, -19.99984167, -13.99989315,   0.        ])

In [21]:
# Test: Make sure the evaluated policy is what we expected
expected_v = np.array([0, -14, -20, -22, -14, -18, -20, -20, -20, -20, -18, -14, -22, -20, -14, 0])
np.testing.assert_array_almost_equal(v, expected_v, decimal=2)